<a href="https://colab.research.google.com/github/mohammedfaizan-max/DIGIBHEM/blob/main/Digital_Bheem_ATM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile User.java
import java.util.ArrayList;

public class User {
    private String userId;
    private String userPin;
    private double balance;
    private ArrayList<String> transactionHistory;

    public User(String userId, String userPin, double initialBalance) {
        this.userId = userId;
        this.userPin = userPin;
        this.balance = initialBalance;
        this.transactionHistory = new ArrayList<>();
        addTransaction("Account opened with initial balance: $" + initialBalance);
    }

    public String getUserId() { return userId; }
    public boolean validatePin(String pin) { return this.userPin.equals(pin); }
    public double getBalance() { return balance; }

    public void deposit(double amount) {
        if (amount > 0) {
            balance += amount;
            addTransaction("Deposited: $" + amount);
        }
    }

    public boolean withdraw(double amount) {
        if (amount > 0 && amount <= balance) {
            balance -= amount;
            addTransaction("Withdrew: $" + amount);
            return true;
        }
        return false;
    }

    public void addTransaction(String message) { transactionHistory.add(message); }
    public ArrayList<String> getTransactionHistory() { return transactionHistory; }
}

Overwriting User.java


In [ ]:
!javac Main.java User.java Transaction.java BankDatabase.java ATM.java

error: file not found: Main.java
Usage: javac <options> <source files>
use --help for a list of possible options


In [ ]:
!javac Main.java User.java Transaction.java BankDatabase.java ATM.java
!java Main

error: file not found: Main.java
Usage: javac <options> <source files>
use --help for a list of possible options
Error: Could not find or load main class Main
Caused by: java.lang.ClassNotFoundException: Main


In [9]:
# 1. Recreate User.java
with open("User.java", "w") as f:
    f.write('''import java.util.ArrayList;
public class User {
    private String userId; private String userPin; private double balance; private ArrayList<String> transactionHistory;
    public User(String userId, String userPin, double initialBalance) {
        this.userId = userId; this.userPin = userPin; this.balance = initialBalance; this.transactionHistory = new ArrayList<>();
        addTransaction("Account opened with initial balance: $" + initialBalance);
    }
    public String getUserId() { return userId; }
    public boolean validatePin(String pin) { return this.userPin.equals(pin); }
    public double getBalance() { return balance; }
    public void deposit(double amount) { if (amount > 0) { balance += amount; addTransaction("Deposited: $" + amount); } }
    public boolean withdraw(double amount) { if (amount > 0 && amount <= balance) { balance -= amount; addTransaction("Withdrew: $" + amount); return true; } return false; }
    public void addTransaction(String message) { transactionHistory.add(message); }
    public ArrayList<String> getTransactionHistory() { return transactionHistory; }
}''')

# 2. Recreate Transaction.java
with open("Transaction.java", "w") as f:
    f.write('''import java.util.ArrayList;
public class Transaction {
    public static void printHistory(User user) {
        System.out.println("\\n======= TRANSACTION HISTORY =======");
        ArrayList<String> history = user.getTransactionHistory();
        if (history.isEmpty()) { System.out.println("No transactions found."); }
        else { for (String record : history) { System.out.println("- " + record); } }
        System.out.println("Current Balance: $" + user.getBalance());
        System.out.println("===================================\\n");
    }
}''')

# 3. Recreate BankDatabase.java
with open("BankDatabase.java", "w") as f:
    f.write('''import java.util.HashMap;
public class BankDatabase {
    private HashMap<String, User> accounts;
    public BankDatabase() {
        accounts = new HashMap<>();
        accounts.put("user123", new User("user123", "4321", 1000.00));
        accounts.put("user567", new User("user567", "9876", 500.00));
    }
    public User getUser(String userId) { return accounts.get(userId); }
    public boolean transferFunds(User sender, String recipientId, double amount) {
        User recipient = getUser(recipientId);
        if (recipient == null) { System.out.println("Error: Recipient not found."); return false; }
        if (sender.getUserId().equals(recipientId)) { System.out.println("Error: Cannot transfer to yourself."); return false; }
        if (sender.withdraw(amount)) { recipient.deposit(amount); sender.addTransaction("Transferred $" + amount + " to: " + recipientId); recipient.addTransaction("Received $" + amount + " from: " + sender.getUserId()); return true; }
        else { System.out.println("Error: Insufficient funds."); return false; }
    }
}''')

# 4. Recreate ATM.java
with open("ATM.java", "w") as f:
    f.write('''import java.util.Scanner;
public class ATM {
    private BankDatabase database; private Scanner scanner;
    public ATM(BankDatabase database) { this.database = database; this.scanner = new Scanner(System.in); }
    public void start() {
        System.out.println("=== WELCOME TO THE DIGITAL BHEM ATM SYSTEM ===");
        System.out.print("Enter User ID: "); String id = scanner.nextLine();
        System.out.print("Enter User PIN: "); String pin = scanner.nextLine();
        User currentUser = database.getUser(id);
        if (currentUser != null && currentUser.validatePin(pin)) { System.out.println("\\n[✔] Login Successful!"); showMenu(currentUser); }
        else { System.out.println("\\n[✘] Invalid Credentials. Exiting."); }
    }
    private void showMenu(User user) {
        boolean quit = false;
        while (!quit) {
            System.out.println("\\n---------- ATM MENU ----------\\n1. Transactions History\\n2. Withdraw\\n3. Deposit\\n4. Transfer\\n5. Quit");
            System.out.print("Select option: "); String choice = scanner.nextLine();
            switch (choice) {
                case "1": Transaction.printHistory(user); break;
                case "2": handleWithdraw(user); break;
                case "3": handleDeposit(user); break;
                case "4": handleTransfer(user); break;
                case "5": System.out.println("\\nGoodbye!"); quit = true; break;
                default: System.out.println("Invalid option.");
            }
        }
    }
    private void handleWithdraw(User user) { System.out.print("Amount: $"); try { double amt = Double.parseDouble(scanner.nextLine()); if (user.withdraw(amt)) System.out.println("Success! Balance: $" + user.getBalance()); else System.out.println("Failed."); } catch(Exception e) { System.out.println("Invalid input."); } }
    private void handleDeposit(User user) { System.out.print("Amount: $"); try { double amt = Double.parseDouble(scanner.nextLine()); if (amt > 0) { user.deposit(amt); System.out.println("Success! Balance: $" + user.getBalance()); } } catch(Exception e) { System.out.println("Invalid input."); } }
    private void handleTransfer(User user) { System.out.print("Recipient ID: "); String rId = scanner.nextLine(); System.out.print("Amount: $"); try { double amt = Double.parseDouble(scanner.nextLine()); if (amt > 0) database.transferFunds(user, rId, amt); } catch(Exception e) { System.out.println("Invalid input."); } }
}''')

# 5. Recreate Main.java
with open("Main.java", "w") as f:
    f.write('''public class Main {
    public static void main(String[] args) {
        BankDatabase db = new BankDatabase();
        ATM atm = new ATM(db);
        atm.start();
    }
}''')

# 6. Compile and Run everything seamlessly
print("--- Compiling Java Source Files ---")
!javac Main.java User.java Transaction.java BankDatabase.java ATM.java
print("--- Running Application ---")
!java Main

--- Compiling Java Source Files ---
--- Running Application ---
=== WELCOME TO THE DIGITAL BHEM ATM SYSTEM ===
Enter User ID: user123
Enter User PIN: 4321

[✔] Login Successful!

---------- ATM MENU ----------
1. Transactions History
2. Withdraw
3. Deposit
4. Transfer
5. Quit
Select option: 3
Amount: $200
Success! Balance: $1200.0

---------- ATM MENU ----------
1. Transactions History
2. Withdraw
3. Deposit
4. Transfer
5. Quit
Select option: 2
Amount: $100
Success! Balance: $1100.0

---------- ATM MENU ----------
1. Transactions History
2. Withdraw
3. Deposit
4. Transfer
5. Quit
Select option: 4
Recipient ID: user567
Amount: $150

---------- ATM MENU ----------
1. Transactions History
2. Withdraw
3. Deposit
4. Transfer
5. Quit
Select option: 1

======= TRANSACTION HISTORY =======
- Account opened with initial balance: $1000.0
- Deposited: $200.0
- Withdrew: $100.0
- Withdrew: $150.0
- Transferred $150.0 to: user567
Current Balance: $950.0


---------- ATM MENU ----------
1. Transactio